# Create Initial/Final Materials

Build an ordered initial → (optional intermediates) → final set of materials from one starting structure, and write them to a subfolder under `uploads/` in path order for [`utils_create_material_set.ipynb`](utils_create_material_set.ipynb).

Nothing here is specific to a Nudged Elastic Band path — that is just the example this notebook is set up for, and the consumer it feeds ([`neb.ipynb`](workflows/neb.ipynb)). Any calculation that takes an ordered start/end pair can use the same output.

Order is preserved by **numbering material names** (`00_...`, `01_...`, …): `utils_create_material_set.ipynb` (and `load_materials_from_folder`) sort by filename, and filenames come from material names.

## Usage

1. Set material and subfolder name in cell 1.2, transform params in 1.3.
1. Run all cells to build and write the path materials.
1. Open [`utils_create_material_set.ipynb`](utils_create_material_set.ipynb), set the same `SUBFOLDER_NAME` and `IS_ORDERED = True`, and run it to save the materials and create the platform set.
1. Use the printed set name as `MATERIAL_SET` in [`neb.ipynb`](workflows/neb.ipynb).

## Summary

1. Install packages and set parameters.
1. Load the starting material — by default the Si(100) surface from Standata.
1. Clone as the initial image; transform a copy into the final image (default: move one surface atom out of the surface plane, into the vacuum).
1. Name members in path order and write them to `uploads/<SUBFOLDER_NAME>/`.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")


### 1.2. Set parameters


In [ ]:
# Starting material (uploads folder or Standata name match)
FOLDER = "uploads"
MATERIAL_NAME = "Silicon (100) surface"

# Short base name for the written images: 00_<PATH_NAME>.json, 01_<PATH_NAME>.json.
# Standata names are long and comma-heavy, and the filename is what sets load order.
PATH_NAME = "Si-100-surface"

# Subfolder under uploads/ to write path materials into — use the same value as
# SUBFOLDER_NAME in utils_create_material_set.ipynb.
SUBFOLDER_NAME = "neb_si_100_surface"


### 1.3. Set example transformation parameters
Default example: move one surface atom out of the surface plane, in crystal coordinates. Replace this cell and 2.3 below for other paths.


In [ ]:
# Atom 5 is the lowest atom of the Si(100) slab, the one facing the vacuum gap.
ATOM_INDEX = 5

# Displacement in Angstrom.
TRANSLATION = [0.0, 0.0, -1.0]


## 2. Build path materials
### 2.1. Load starting material


In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

source_material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME)
)
visualize(source_material)


### 2.2. Clone as the initial image


In [ ]:
initial_material = source_material.clone()
visualize(initial_material)


### 2.3. Transform into the final image (example)

Default: move one surface atom by `TRANSLATION`, given in Ångström. Replace with any other transformation (defects, swaps, custom coordinates, …) — only the resulting list of materials in 2.4 matters.

For a smooth localized displacement rather than a single-atom jump, `mat3ra.made.tools.helpers.create_perturbation(material, expression, use_cartesian_coordinates=False)` maps a scalar `f(x, y, z)` to `∆z`; a Gaussian centred on the moving atom gives the same path with its neighbours relaxing along it.


In [ ]:
from mat3ra.notebooks_utils.material import translate_atoms

final_material = translate_atoms(initial_material, ATOM_INDEX, TRANSLATION)

print(
    f"Atom {ATOM_INDEX}: {initial_material.coordinates_array[ATOM_INDEX]} → "
    f"{final_material.coordinates_array[ATOM_INDEX]}"
)
visualize([initial_material, final_material])


### 2.4. Name members in path order and write to the subfolder

Numeric prefixes control load order in `utils_create_material_set.ipynb` (filenames are sorted; filenames come from material names).


In [ ]:
from mat3ra.notebooks_utils.material import set_materials
from mat3ra.notebooks_utils.settings import UPLOADS_FOLDER

path_materials = [initial_material, final_material]
for index, material in enumerate(path_materials):
    material.name = f"{index:02d}_{PATH_NAME}"

subfolder_path = f"{UPLOADS_FOLDER}/{SUBFOLDER_NAME}"
set_materials(path_materials, folder_path=subfolder_path)
